In [25]:
from scipy.stats import friedmanchisquare
import pandas as pd
import numpy as np

from scipy.stats import wilcoxon
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# Simu-A1

In [26]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Data_0224/Results_SimulationA1_sig0.2.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_SimulationA1_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   624.0592  2.7322E-123  Significant differences exist
1  Test_MAE   458.3261   3.6058E-88  Significant differences exist
2   Test_R2   561.2642  5.9420E-110  Significant differences exist

Saved:
Results_SimulationA1_Friedman.xlsx


In [28]:
FKAN_name='F-KAN'

output_file='Results_SimulationA1_sig0.2_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_SimulationA1_sig0.2_Wilcoxon.xlsx


**SimuA2**

In [29]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Data_0224/Results_SimulationA2_sig0.2.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_SimulationA2_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   636.8538  5.1917E-126  Significant differences exist
1  Test_MAE  1070.7736  8.9942E-219  Significant differences exist
2   Test_R2   849.5971  2.1383E-171  Significant differences exist

Saved:
Results_SimulationA2_Friedman.xlsx


In [30]:
FKAN_name='F-KAN'

output_file='Results_SimulationA2_sig0.2_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_SimulationA2_sig0.2_Wilcoxon.xlsx


# SimuB1

In [31]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Data_0224/Results_SimulationB1_sig0.2.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_SimulationB1_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   875.7299  5.5047E-177  Significant differences exist
1  Test_MAE   824.2108  5.7175E-166  Significant differences exist
2   Test_R2   962.7028  1.3228E-195  Significant differences exist

Saved:
Results_SimulationB1_Friedman.xlsx


In [32]:
FKAN_name='F-KAN'

output_file='Results_SimulationB1_sig0.2_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_SimulationB1_sig0.2_Wilcoxon.xlsx


# SimuB2

In [33]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Data_0224/Results_SimulationB2_sig0.2.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_SimulationB2_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   807.3369  2.3072E-162  Significant differences exist
1  Test_MAE   931.0593  7.9187E-189  Significant differences exist
2   Test_R2   933.7353  2.1167E-189  Significant differences exist

Saved:
Results_SimulationB2_Friedman.xlsx


In [34]:
FKAN_name='F-KAN'

output_file='Results_SimulationB2_sig0.2_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_SimulationB2_sig0.2_Wilcoxon.xlsx


# CGL

In [35]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Real_0226/Results_CGL.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_CGL_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   696.3998  1.0880E-138  Significant differences exist
1  Test_MAE   700.4748  1.4729E-139  Significant differences exist
2   Test_R2   695.7245  1.5154E-138  Significant differences exist

Saved:
Results_CGL_Friedman.xlsx


In [36]:
FKAN_name='F-KAN'

output_file='Results_CGL_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_CGL_Wilcoxon.xlsx


# Tecator

In [37]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Real_0226/Results_Tecator.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_Tecator_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic     P-value                     Conclusion
0  Test_MSE   357.6555  5.2575E-67  Significant differences exist
1  Test_MAE   423.6519  7.3345E-81  Significant differences exist
2   Test_R2   341.4832  1.2668E-63  Significant differences exist

Saved:
Results_Tecator_Friedman.xlsx


In [38]:
FKAN_name='F-KAN'

output_file='Results_Tecator_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_Tecator_Wilcoxon.xlsx


# AA

In [39]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Real_0226/Results_AA.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_AA_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   616.4946  1.1088E-121  Significant differences exist
1  Test_MAE   668.3553  1.0250E-132  Significant differences exist
2   Test_R2   618.8788  3.4513E-122  Significant differences exist

Saved:
Results_AA_Friedman.xlsx


In [40]:
FKAN_name='F-KAN'

output_file='Results_AA_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_AA_Wilcoxon.xlsx


# Milk

In [41]:
# Excel文件
file_path='/kaggle/input/datasets/fireflyqmf313/f-kan-results-plot/Real_0226/Results_Milk.xlsx'

# 获取所有sheet名（每个sheet=一个模型）
xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names

metrics=[

    'Test_MSE',
    'Test_MAE',
    'Test_R2'

]

xls=pd.ExcelFile(file_path)

sheet_names=xls.sheet_names



friedman_results=[]


for metric in metrics:

    all_results=[]

    for sheet in sheet_names:

        df=pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        values=df[metric].iloc[2:102]

        values=values.reset_index(drop=True)

        all_results.append(values)

    stat,p=friedmanchisquare(*all_results)

    if p<0.05:

        conclusion='Significant differences exist'

    else:

        conclusion='No significant difference'

    friedman_results.append([

        metric,

        round(stat,4),

        "{:.4E}".format(p),

        conclusion

    ])


result_df=pd.DataFrame(

    friedman_results,

    columns=[

        'Metric',

        'Statistic',

        'P-value',

        'Conclusion'

    ]

)


print('\nFriedman Results:\n')

print(result_df)


output='Results_Milk_Friedman.xlsx'

result_df.to_excel(

    output,

    index=False

)

print('\nSaved:')

print(output)


Friedman Results:

     Metric  Statistic      P-value                     Conclusion
0  Test_MSE   296.2113   3.4235E-54  Significant differences exist
1  Test_MAE   521.6993  1.4446E-101  Significant differences exist
2   Test_R2   273.3385   1.8881E-49  Significant differences exist

Saved:
Results_Milk_Friedman.xlsx


In [42]:
FKAN_name='F-KAN'

output_file='Results_Milk_Wilcoxon.xlsx'

all_data={}

for model in sheet_names:

    df=pd.read_excel(
        file_path,
        sheet_name=model
    )

    temp={}

    for metric in metrics:

        # 第2行为均值
        # 第3~102行为100次实验结果

        values=df[metric].iloc[2:102]

        values=values.astype(float)

        values=values.reset_index(drop=True)

        temp[metric]=values

    all_data[model]=temp


########################################################
# 输出多个sheet
########################################################

with pd.ExcelWriter(
        output_file,
        engine='openpyxl'
) as writer:


    for metric in metrics:

        fkan=all_data[FKAN_name][metric]

        results=[]

        pvals=[]


        ################################################
        # F-KAN vs others
        ################################################

        for model in sheet_names:

            if model==FKAN_name:
                continue


            x=all_data[model][metric]


            ############################################
            # Wilcoxon
            ############################################

            stat,p=wilcoxon(

                fkan,
                x,
                alternative='two-sided'

            )


            ############################################
            # 效应量
            ############################################

            n=len(fkan)

            mean_w=n*(n+1)/4

            std_w=np.sqrt(
                n*(n+1)*(2*n+1)/24
            )

            z=(stat-mean_w)/std_w

            r=abs(z)/np.sqrt(n)


            ############################################
            # 方向
            ############################################

            fkan_mean=np.mean(fkan)

            x_mean=np.mean(x)


            if metric in ['Test_MSE','Test_MAE']:

                if fkan_mean<x_mean:

                    direction='F-KAN better'

                elif fkan_mean>x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            elif metric=='Test_R2':

                if fkan_mean>x_mean:

                    direction='F-KAN better'

                elif fkan_mean<x_mean:

                    direction='F-KAN worse'

                else:

                    direction='Equal'


            ############################################

            results.append([

                f'F-KAN vs {model}',

                direction,

                stat,

                p,

                r

            ])


            pvals.append(p)


        ############################################
        # Holm correction
        ############################################

        reject,p_adjust,_,_=multipletests(

            pvals,

            alpha=0.05,

            method='holm'

        )


        ############################################
        # DataFrame
        ############################################

        result_df=pd.DataFrame(

            results,

            columns=[

                'Comparison',

                'Direction',

                'Statistic',

                'P_raw',

                'Effect_size'

            ]

        )


        result_df['P_adjust']=p_adjust

        result_df['Significant']=reject


        ############################################
        # 排列列顺序
        ############################################

        result_df=result_df[[

            'Comparison',

            'Direction',

            'Statistic',

            'P_raw',

            'P_adjust',

            'Effect_size',

            'Significant'

        ]]


        ############################################
        # 保存sheet
        ############################################

        result_df.to_excel(

            writer,

            sheet_name=metric,

            index=False

        )


print('\nDone.')

print(output_file)


Done.
Results_Milk_Wilcoxon.xlsx
